In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time
import datetime as dt
from selenium.webdriver.common.by import By
import datetime
from datetime import date
from datetime import datetime, timedelta
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
#Open Chrome
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

import pandas as pd
import gspread
import pygsheets
import numpy as np

In [6]:
chrome_options = Options()
chrome_options.add_argument("--remote-debugging-port=9222")
chrome_options.add_argument('user-data-dir=C:\\selenium\\ChromeProfile')
chrome_driver = "C:\\Users\\BSS\\BSS\\chromedriver"
driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()), options=chrome_options)
driver.maximize_window()

In [7]:
client = pygsheets.authorize(service_account_file="xxxxxxxxxxxx.json")
sh=client.open('Linh - Full Shopify App Market Research')
wks = sh.worksheet_by_title('Competitor List') # open the existing file
cells = wks.get_all_values(include_tailing_empty_rows=False, include_tailing_empty=False, returnas='matrix')
df_get_link = pd.DataFrame(cells)

#set first row as column
df_get_link.columns = df_get_link.iloc[0]
df_get_link = df_get_link[1:]

#list of competitors' links
all_competitor_link = df_get_link['Competitor Link'].values.tolist()
all_competitor_link = [i for i in all_competitor_link if i is not None]
len(all_competitor_link)

160

In [8]:
review_sh = client.open("Competitor's Reviews")
competitor_list_wks = review_sh.worksheet_by_title("Competitors List") # open the existing file
competitor_list_cells = competitor_list_wks.get_all_values(include_tailing_empty_rows=False, include_tailing_empty=False, returnas='matrix')
df_get_link2 = pd.DataFrame(competitor_list_cells)
#set first row as column
df_get_link2.columns = df_get_link2.iloc[0]
df_get_link2 = df_get_link2[1:]


#list of old competitors' links
old_competitor_link = df_get_link2['App Link'].values.tolist()
len(old_competitor_link)

154

In [24]:
reviews_list = []

In [10]:
#page_of_review: là page chứa các review (để check có bao nhiêu reviews trong page đó)
for link in old_competitor_link[75:]:
    driver.get(link)
    try:
        brand = driver.find_element(By.XPATH,'//*[@id="adp-hero"]/div/div/div[1]/div/div[1]/div[2]/div[2]/div[3]/div').text
    except:
        continue
    time.sleep(2)
    count = 1
    while True:    
        review_link = link + '/reviews?sort_by=newest&page={}'.format(str(count))
        driver.get(review_link)
        time.sleep(1)
        page_of_review = []
        for i in range(1,11):
            try:
                reviews = driver.find_element(By.XPATH,'//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]'.format(i))
                page_of_review.append(reviews)
            except:
                continue
        if len(page_of_review) == 0:
            break
        try:                                    
            app = driver.find_element(By.XPATH,'//*[@id="adp-reviews"]/div/div[1]/div[1]/h1/span[1]/a/span').text
            for Review in page_of_review:
                index = page_of_review.index(Review)
                name = Review.find_element(By.XPATH,'//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]/div[3]'.format(str(index+1))).text
                location = Review.find_element(By.XPATH,'//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]/div[4]/span'.format(str(index+1))).text
                try:                                    
                    time_spent= Review.find_element(By.XPATH,'//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]/div[4]/div/span'.format(str(index+1))).text
                except:
                    time_spent = 'Time spent using app: 0 days'
                review_full_content = Review.find_element(By.XPATH, '//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]/div[2]/div'.format(str(index+1))).text
                rating = Review.find_element(By.XPATH, '//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]/div[1]/div[1]'.format(str(index+1))).get_attribute("aria-label")
                review_date = Review.find_element(By.XPATH, '//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]/div[1]/div[2]'.format(str(index+1))).text
                Review_item = { 
                            "Brand Name": brand,
                            'App Name': app,
                            'App Link': link,
                            'Review Link': review_link,
                            'Reviewer name': name,
                            'Reviewer location': location,
                            'Time spent on App (day)': time_spent,
                            'Review full content': review_full_content,
                            'Rating': rating,
                            'Review date': review_date,
                        }
                reviews_list.append(Review_item)
        except:
            pass
        count += 1
        if count > 25:
            break

In [11]:
new_competitor_list = [x for x in all_competitor_link if x not in old_competitor_link]

In [25]:
#page_of_review: là page chứa các review (để check có bao nhiêu reviews trong page đó)
for link in new_competitor_list:
    driver.get(link)
    try:
        brand = driver.find_element(By.XPATH,'//*[@id="adp-hero"]/div/div/div[1]/div/div[1]/div[2]/div[2]/div[3]/div/a').text
    except:
        continue
    if len(brand) == 0:
        break
    time.sleep(2)
    count = 1
    while True:    
        review_link = link + '/reviews?sort_by=newest&page={}'.format(str(count))
        driver.get(review_link)
        
        page_of_review = []
        for i in range(1,11):
            try:
                reviews = driver.find_element(By.XPATH,'//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]'.format(i))
                page_of_review.append(reviews)
            except:
                continue
        if len(page_of_review) == 0:
            break
        try:                                 
            app = driver.find_element(By.XPATH,'//*[@id="adp-reviews"]/div/div[1]/div[1]/h1/span[1]/a/span').text
            for Review in page_of_review:
                index = page_of_review.index(Review)
                name = Review.find_element(By.XPATH,'//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]/div[3]'.format(str(index+1))).text
                location = Review.find_element(By.XPATH,'//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]/div[4]/span'.format(str(index+1))).text
                try:                                    
                    time_spent= Review.find_element(By.XPATH,'//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]/div[4]/div/span'.format(str(index+1))).text
                except:
                    time_spent = 'Time spent using app: 0 days'
                review_full_content = Review.find_element(By.XPATH, '//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]/div[2]/div'.format(str(index+1))).text
                rating = Review.find_element(By.XPATH, '//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]/div[1]/div[1]'.format(str(index+1))).get_attribute("aria-label")
                review_date = Review.find_element(By.XPATH, '//*[@id="adp-reviews"]/div/div[2]/div[3]/div[1]/div[{}]/div[1]/div[1]/div[2]'.format(str(index+1))).text
                Review_item = { 
                            "Brand Name": brand,
                            'App Name': app,
                            'App Link': link,
                            'Review Link': review_link,
                            'Reviewer name': name,
                            'Reviewer location': location,
                            'Time spent on App (day)': time_spent,
                            'Review full content': review_full_content,
                            'Rating': rating,
                            'Review date': review_date,
                        }
                reviews_list.append(Review_item)
        except:
            pass
        count += 1

In [26]:
df = pd.DataFrame(reviews_list)
df

,Brand Name,App Name,App Link,Review Link,Reviewer name,Reviewer location,Time spent on App (day),Review full content,Rating,Review date
0,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,FABINALIV,India,Time spent using app: 26 minutes,Very helpful and understanding team. They help...,5 out of 5 stars,"July 3, 2023"
1,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,Merakrt,India,Time spent using app: 0 days,"Best App for Popups for E-commerce website, Th...",5 out of 5 stars,"July 1, 2023"
2,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,balevski,Bulgaria,Time spent using app: 0 days,"great features, great service, everything happ...",5 out of 5 stars,"June 30, 2023"
3,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,Home.Vip,Brazil,Time spent using app: 15 days,"curti o app, dá pra personalizar os popups e a...",5 out of 5 stars,"June 18, 2023"
4,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,brazilmulticosmetics,Brazil,Time spent using app: 1 minute,"very good app, I recommend the app, it makes t...",5 out of 5 stars,"May 29, 2023"
...,...,...,...,...,...,...,...,...,...,...
1703,qikify,qikify Promotion Bar & Banner,https://apps.shopify.com/smart-bar,https://apps.shopify.com/smart-bar/reviews?sor...,Newverest,United States,Time spent using app: 4 months,I love this app. Help me generated more than $...,5 out of 5 stars,"Edited November 27, 2018"
1704,qikify,qikify Promotion Bar & Banner,https://apps.shopify.com/smart-bar,https://apps.shopify.com/smart-bar/reviews?sor...,Michael Malta Studio,United States,Time spent using app: 10 months,This is a fantastic plugin. It just works. Gre...,5 out of 5 stars,"Edited September 9, 2019"
1705,qikify,qikify Promotion Bar & Banner,https://apps.shopify.com/smart-bar,https://apps.shopify.com/smart-bar/reviews?sor...,Luxury-Clock,France,Time spent using app: 5 months,"Très personnalisable,très complet, et support ...",5 out of 5 stars,"November 6, 2018"
1706,qikify,qikify Promotion Bar & Banner,https://apps.shopify.com/smart-bar,https://apps.shopify.com/smart-bar/reviews?sor...,LUCY LUE ORGANICS,United States,Time spent using app: 5 days,We love this app! It is so easy to use! The op...,5 out of 5 stars,"Edited October 25, 2018"


In [27]:
def days_spent(time):
    days_spent =  [int(s) for s in time.split() if s.isdigit()][0]
    dict_time = {"minute": 1/(60*24), "hour": 1/24, "day": 1, "month": 30, "year": 365}
    for unit in dict_time:
        if unit in time:
            days_spent_clean = days_spent*dict_time[unit]
            if "Over" in time: days_spent_clean=days_spent*dict_time[unit]*1.01
            if"About" in time: days_spent_clean=days_spent*dict_time[unit]*0.99
            if"Almost" in time: days_spent_clean=days_spent*dict_time[unit]*0.99
            
    return days_spent_clean

In [28]:
df["Time spent on App (day)"] = df["Time spent on App (day)"].astype(str).apply(days_spent)

#Get Only Rate:
df['Rating'] = df['Rating'].str.replace(r' out of 5 stars', '').astype(int)

#Remove 'Edit'
for x in df.index:
    if "Edited" in df.loc[x, 'Review date']:
        df.loc[x, 'Review date']=df.loc[x, 'Review date'].replace("Edited ","")
df["Review Month"] = pd.to_datetime(df['Review date']).dt.to_period('M')
df["Review Month"] = df["Review Month"].dt.strftime('%b %Y')
df["Review date"] = pd.to_datetime(df["Review date"],format='%B %d, %Y') 
df["Review date"] = df["Review date"].dt.strftime('%m/%d/%Y')
df

,Brand Name,App Name,App Link,Review Link,Reviewer name,Reviewer location,Time spent on App (day),Review full content,Rating,Review date,Review Month
0,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,FABINALIV,India,0.018056,Very helpful and understanding team. They help...,5,07/03/2023,Jul 2023
1,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,Merakrt,India,0.000000,"Best App for Popups for E-commerce website, Th...",5,07/01/2023,Jul 2023
2,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,balevski,Bulgaria,0.000000,"great features, great service, everything happ...",5,06/30/2023,Jun 2023
3,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,Home.Vip,Brazil,15.000000,"curti o app, dá pra personalizar os popups e a...",5,06/18/2023,Jun 2023
4,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,brazilmulticosmetics,Brazil,0.000694,"very good app, I recommend the app, it makes t...",5,05/29/2023,May 2023
...,...,...,...,...,...,...,...,...,...,...,...
1703,qikify,qikify Promotion Bar & Banner,https://apps.shopify.com/smart-bar,https://apps.shopify.com/smart-bar/reviews?sor...,Newverest,United States,120.000000,I love this app. Help me generated more than $...,5,11/27/2018,Nov 2018
1704,qikify,qikify Promotion Bar & Banner,https://apps.shopify.com/smart-bar,https://apps.shopify.com/smart-bar/reviews?sor...,Michael Malta Studio,United States,300.000000,This is a fantastic plugin. It just works. Gre...,5,09/09/2019,Sep 2019
1705,qikify,qikify Promotion Bar & Banner,https://apps.shopify.com/smart-bar,https://apps.shopify.com/smart-bar/reviews?sor...,Luxury-Clock,France,150.000000,"Très personnalisable,très complet, et support ...",5,11/06/2018,Nov 2018
1706,qikify,qikify Promotion Bar & Banner,https://apps.shopify.com/smart-bar,https://apps.shopify.com/smart-bar/reviews?sor...,LUCY LUE ORGANICS,United States,5.000000,We love this app! It is so easy to use! The op...,5,10/25/2018,Oct 2018


In [29]:
review_sh = client.open("Competitor's Reviews")
review_wks = review_sh.worksheet_by_title("Competitor's Reviews Data") # open the existing file
review_cells = review_wks.get_all_values(include_tailing_empty_rows=False, include_tailing_empty=False, returnas='matrix')
check_duplicates = pd.DataFrame(review_cells)

#set first row as column
check_duplicates.columns = check_duplicates.iloc[0]
check_duplicates = check_duplicates[1:]

check_duplicates.drop(["BSS App","Brand Name", "App Name", "Review Link", "Reviewer name","Reviewer location","Time spent on App (day)","Rating","Review Month","Unique App Name"], axis=1, inplace=True)
check_duplicates

,App Link,Review full content,Review date
1,https://apps.shopify.com/product-labels-badges,Exactly what I was missing. Support is very re...,12/14/2022
2,https://apps.shopify.com/product-labels-badges,This app is so easy to use but support chat is...,12/12/2022
3,https://apps.shopify.com/product-labels-badges,One of the best app that I'm using.\nAnd the s...,11/30/2022
4,https://apps.shopify.com/product-labels-badges,Super powerful customer support app. We recomm...,11/24/2022
5,https://apps.shopify.com/product-labels-badges,"For its purpose, Labeler is a great app! It al...",11/23/2022
...,...,...,...
99885,https://apps.shopify.com/smart-bar,I love this app. Help me generated more than $...,11/27/2018
99886,https://apps.shopify.com/smart-bar,This is a fantastic plugin. It just works. Gre...,9/9/2019
99887,https://apps.shopify.com/smart-bar,"Très personnalisable,très complet, et support ...",11/6/2018
99888,https://apps.shopify.com/smart-bar,We love this app! It is so easy to use! The op...,10/25/2018


In [30]:
df = pd.merge(df, check_duplicates, on=["App Link","Review full content","Review date"], how="outer", indicator=True)
df = df.loc[df["_merge"] == "left_only"].drop(["_merge"], axis=1)
df.drop_duplicates(inplace=True)
df

,Brand Name,App Name,App Link,Review Link,Reviewer name,Reviewer location,Time spent on App (day),Review full content,Rating,Review date,Review Month
0,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,FABINALIV,India,0.018056,Very helpful and understanding team. They help...,5.0,07/03/2023,Jul 2023
1,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,Merakrt,India,0.000000,"Best App for Popups for E-commerce website, Th...",5.0,07/01/2023,Jul 2023
2,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,balevski,Bulgaria,0.000000,"great features, great service, everything happ...",5.0,06/30/2023,Jun 2023
3,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,Home.Vip,Brazil,15.000000,"curti o app, dá pra personalizar os popups e a...",5.0,06/18/2023,Jun 2023
4,MakeProSimp,ToastiBar ‑ Sales Popup,https://apps.shopify.com/mps-sales-notification,https://apps.shopify.com/mps-sales-notificatio...,brazilmulticosmetics,Brazil,0.000694,"very good app, I recommend the app, it makes t...",5.0,05/29/2023,May 2023
...,...,...,...,...,...,...,...,...,...,...,...
1703,qikify,qikify Promotion Bar & Banner,https://apps.shopify.com/smart-bar,https://apps.shopify.com/smart-bar/reviews?sor...,Dgitrends,United States,300.000000,I love this app. People can't seem to keep fro...,5.0,03/14/2019,Mar 2019
1704,qikify,qikify Promotion Bar & Banner,https://apps.shopify.com/smart-bar,https://apps.shopify.com/smart-bar/reviews?sor...,Madera Case,Australia,29.700000,Super easy to use tool - does exactly what it ...,5.0,01/30/2019,Jan 2019
1706,qikify,qikify Promotion Bar & Banner,https://apps.shopify.com/smart-bar,https://apps.shopify.com/smart-bar/reviews?sor...,Michael Malta Studio,United States,300.000000,This is a fantastic plugin. It just works. Gre...,5.0,09/09/2019,Sep 2019
1707,qikify,qikify Promotion Bar & Banner,https://apps.shopify.com/smart-bar,https://apps.shopify.com/smart-bar/reviews?sor...,Luxury-Clock,France,150.000000,"Très personnalisable,très complet, et support ...",5.0,11/06/2018,Nov 2018


In [31]:
review_sh = client.open("Competitor's Reviews")
review_wks = review_sh.worksheet_by_title("Competitor's Reviews Data") # open the existing file
review_cells = review_wks.get_all_values(include_tailing_empty_rows=False, include_tailing_empty=False, returnas='matrix')
review_wks.set_dataframe(df,(len(review_cells)+1,2),copy_head = False ,extend = True)